# 06 Pooling And Representation

Reviewer-facing pooling sensitivity and representation-evidence status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or was stale: {exc}')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
        if os.environ.get('ECG_RAMBA_ALLOW_STALE_REPO', '0') != '1':
            raise RuntimeError('git pull failed; refusing to continue with stale code. Set ECG_RAMBA_ALLOW_STALE_REPO=1 only for debugging.')
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_versioned_git_release_v2'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 2
_AUTHORITY_BOOTSTRAP_ALLOWED = False
_AUTHORITY_DEFAULT_RELEASE_REF = 'refs/tags/ecg-ramba-revision-20260722-v5'

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath
    from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    requested_release_ref = _authority_os.environ.get(
        'ECG_RAMBA_AUTHORITY_REF', _AUTHORITY_DEFAULT_RELEASE_REF
    ).strip()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    legacy_rotate_requested = (
        _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
    )
    if legacy_rotate_requested:
        raise RuntimeError(
            'Implicit authority rotation to a moving branch head is disabled. '
            'Notebook 00 follows only a reviewed versioned release tag. For an emergency explicit '
            'pin, set ECG_RAMBA_RESET_CODE_AUTHORITY=1 with a full ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    rotate_to_branch_head = False
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')
    release_ref_pattern = _authority_re.compile(r'refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    def resolve_annotated_release_ref(release_ref):
        if not release_ref_pattern.fullmatch(release_ref):
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_REF must be a full refs/tags/... name. '
                'Moving branches are not accepted as code authority.'
            )
        object_type = git('cat-file', '-t', release_ref, check=False)
        if object_type.returncode or object_type.stdout.strip() != 'tag':
            raise RuntimeError(
                f'Code-authority release ref is missing or is not an annotated Git tag: {release_ref}. '
                'Run the current Notebook 00 after the reviewed release tag has been pushed.'
            )
        object_id = git('rev-parse', '--verify', release_ref).stdout.strip().lower()
        commit = git('rev-parse', '--verify', release_ref + '^{commit}').stdout.strip().lower()
        if not commit_pattern.fullmatch(object_id) or not commit_pattern.fullmatch(commit):
            raise RuntimeError(f'Could not resolve an immutable object and commit for {release_ref}.')
        return commit, object_id

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', '--tags', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit/release tag.')
        print((fetch.stdout or '')[-2000:])

    manifest = None
    manifest_raw = None
    with _AuthorityPublishLock(
        _AuthorityPath(MIRROR_REVISION_ROOT),
        run_id='authority-read-' + _authority_uuid.uuid4().hex,
    ):
        if manifest_path.is_file():
            manifest_raw = manifest_path.read_text(encoding='utf-8')
            manifest = _authority_json.loads(manifest_raw)

    authority_update_needed = False
    update_reason = None
    release_ref = None
    release_ref_object_id = None

    if reset_requested:
        expected_commit = requested_commit
        authority_update_needed = True
        update_reason = 'explicit_environment_sha'
    elif manifest is None:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        release_ref = requested_release_ref
        expected_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
        authority_update_needed = True
        update_reason = 'initial_versioned_release_bootstrap'
    else:
        manifest_capability = manifest.get('capability')
        manifest_schema = int(manifest.get('schema_version', 0))
        legacy_manifest = (
            manifest_capability == 'canonical_git_commit_pin_v1' and manifest_schema == 1
        )
        current_manifest = (
            manifest_capability == 'canonical_versioned_git_release_v2' and manifest_schema == 2
        )
        if not legacy_manifest and not current_manifest:
            raise RuntimeError('Canonical code-authority manifest capability/schema is invalid.')
        manifest_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(manifest_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')

        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            if legacy_manifest:
                raise RuntimeError(
                    'Canonical code authority uses the legacy schema. Run the current Notebook 00 once '
                    'to migrate it to the reviewed versioned release before running downstream notebooks.'
                )
            expected_commit = manifest_commit
            if requested_commit and requested_commit != expected_commit:
                raise RuntimeError(
                    'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                    'Rotate authority explicitly in Notebook 00; do not override it downstream.'
                )
        else:
            release_ref = requested_release_ref
            release_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
            manifest_ref = str(manifest.get('authority_ref', '')).strip()
            manifest_ref_object = str(manifest.get('authority_ref_object_id', '')).strip().lower()
            if current_manifest and manifest_ref == release_ref:
                if manifest_commit != release_commit or manifest_ref_object != release_ref_object_id:
                    raise RuntimeError(
                        f'Code-authority release tag moved or changed after it was recorded: {release_ref}. '
                        'Publish a new versioned tag instead of retagging an existing release.'
                    )
                expected_commit = manifest_commit
            else:
                expected_commit = release_commit
                authority_update_needed = True
                update_reason = (
                    'legacy_manifest_migration'
                    if legacy_manifest
                    else 'versioned_release_upgrade'
                )

    if not commit_pattern.fullmatch(expected_commit):
        raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if authority_update_needed:
        previous_commit = None if manifest is None else str(manifest.get('git_commit', '')).strip().lower()
        previous_ref = None if manifest is None else manifest.get('authority_ref')
        manifest = {
            'capability': 'canonical_versioned_git_release_v2',
            'schema_version': 2,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'explicit_environment_sha'
                if release_ref is None
                else 'verified_annotated_versioned_release_tag'
            ),
            'authority_ref': release_ref,
            'authority_ref_kind': None if release_ref is None else 'annotated_git_tag',
            'authority_ref_object_id': release_ref_object_id,
            'update_reason': update_reason,
            'previous_git_commit': previous_commit,
            'previous_authority_ref': previous_ref,
        }
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-write-' + _authority_uuid.uuid4().hex,
        ):
            concurrent_raw = manifest_path.read_text(encoding='utf-8') if manifest_path.is_file() else None
            if concurrent_raw != manifest_raw and not reset_requested:
                concurrent = _authority_json.loads(concurrent_raw) if concurrent_raw else None
                if not (
                    concurrent
                    and concurrent.get('capability') == 'canonical_versioned_git_release_v2'
                    and int(concurrent.get('schema_version', 0)) == 2
                    and str(concurrent.get('git_commit', '')).lower() == expected_commit
                    and concurrent.get('authority_ref') == release_ref
                    and str(concurrent.get('authority_ref_object_id', '')).lower()
                    == str(release_ref_object_id or '').lower()
                ):
                    raise RuntimeError('A concurrent runtime established a different code authority.')
                manifest = concurrent
            else:
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = manifest_path.with_name(
                    manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                )
                with temporary.open('w', encoding='utf-8') as handle:
                    handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                    handle.flush()
                    _authority_os.fsync(handle.fileno())
                _authority_os.replace(temporary, manifest_path)
        print('Established/updated canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority release  :', manifest.get('authority_ref'))
    print('Authority manifest :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN

_run_setup('git status --short --branch', check=False)

DIRECT_RUN_SOURCE_REQUIREMENTS_06 = {
    'scripts/revision/artifact_mirror.py': ['--verify-existing', '--include-prefix'],
    'scripts/revision/20_representation_probe.py': [
        '--probe-max-iter', '--out-fold-probe-table', '--out-audit-figure',
        'representation_probe_fold_safe_v3_projection_and_fold_audit',
    ],
    'scripts/revision/22_extract_representations.py': [
        '--fold-cache-dir', 'canonical_contract', 'embedding_npz_sha256',
    ],
    'scripts/revision/30_pooling_sensitivity_external.py': [
        'external_pooling_sensitivity_v2_group_bootstrap', '--strict-group-bootstrap',
    ],
    'scripts/revision/44_physiological_interval_probe.py': [
        'fold_held_out_measured_physiological_interval_probe_v3', 'RUNNER_SOURCE_PATH',
        '--embedding-manifest', 'independent_of_model_outputs',
        'independent_of_ecg_ramba_feature_cache', 'metadata_sha256',
    ],
}
source_failures_06 = []
for relative, tokens in DIRECT_RUN_SOURCE_REQUIREMENTS_06.items():
    source_path = REPO_DIR / relative
    source_text = source_path.read_text(encoding='utf-8', errors='replace') if source_path.exists() else ''
    missing_tokens = [token for token in tokens if token not in source_text]
    if missing_tokens:
        source_failures_06.append(f'{relative}: missing {missing_tokens}')
if source_failures_06:
    raise RuntimeError(
        'Notebook 06 checked out stale representation code. Pull/push latest main before GPU extraction: '
        + ' ; '.join(source_failures_06)
    )
print('Notebook 06 direct-run source preflight: OK')

# Direct-run CPU dependencies. The Mamba/CUDA runtime is installed lazily only if
# the authenticated embedding package is missing or stale.
INSTALL_NOTEBOOK06_BASE_DEPS = os.environ.get('ECG_RAMBA_INSTALL_NOTEBOOK06_BASE_DEPS', '1') == '1'
if INSTALL_NOTEBOOK06_BASE_DEPS:
    _run_setup(
        'python -m pip install -q ' 
        '"numpy>=2.0,<2.1" "scipy>=1.14.1,<2.0" "pandas==2.2.2" ' 
        '"scikit-learn>=1.2,<1.9" joblib tqdm packaging matplotlib seaborn umap-learn',
        cwd=REPO_DIR,
    )
else:
    print('Skipping Notebook 06 base dependency install because ECG_RAMBA_INSTALL_NOTEBOOK06_BASE_DEPS=0')

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)

    from contextlib import ExitStack
    with ExitStack() as stack:
        log_file = stack.enter_context(log_path.open('a', encoding='utf-8'))
        durable_log_file = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
            durable_log_file.write(line)
            durable_log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    print(f'Durable command log: {durable_log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Scope And Contracts

This notebook has two reviewer-facing analyses:

- Pooling sensitivity from checksum-verified frozen OOF slice probabilities.
- Optional representation embedding extraction and probe/CKA analysis from frozen final_ema checkpoints.

Representation evidence remains claim-limited: even when probes run, wording must be "suggestive branch-specific information", not proven morphology-rhythm disentanglement.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

THRESHOLD = 0.5
RUN_POOLING_SENSITIVITY = True
FORCE_RERUN_POOLING_SENSITIVITY = False
RUN_REPRESENTATION_EMBEDDING_EXTRACTION = 'auto'
RUN_REPRESENTATION_PROBING = 'auto'
REPRESENTATION_ONLY_FOLDS = ''  # Example: '1,2' for interrupted Colab sessions; empty means all/cache assemble.
REPRESENTATION_BATCH_SIZE = 64
REPRESENTATION_NUM_WORKERS = 0
REPRESENTATION_FOLD_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'predictions' / 'folds'
REPRESENTATION_FOLD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPRESENTATION_PROBE_MAX_ITER = 5000
OOF_STEM = 'oof_final_ema'

revision_root = Path('reports/revision')
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
record_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
slice_path = revision_root / 'predictions' / f'{OOF_STEM}_slice_predictions.npz'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
run_manifest_path = revision_root / 'manifests' / f'{OOF_STEM}_prediction_run_manifest.json'

RESTORE_RELATIVE_PATHS_FOR_06 = [
    'manifests/oof_final_ema_freeze_manifest.json',
    'manifests/oof_final_ema_prediction_run_manifest.json',
    'predictions/oof_final_ema_predictions.npz',
    'predictions/oof_final_ema_slice_predictions.npz',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'metrics/pooling_sensitivity.csv',
    'metrics/pooling_sensitivity.json',
    'metrics/pooling_decision_summary.json',
    'tables/table_pooling_sensitivity.csv',
    'metrics/pooling_sensitivity_external.csv',
    'tables/table_pooling_sensitivity_across_datasets.csv',
    'metrics/pooling_q3_paired_bootstrap.json',
    'manifests/pooling_sensitivity_external_manifest.json',
    'metrics/external_ptbxl_protocol_gate.json',
    'metrics/external_georgia_protocol_gate.json',
    'metrics/external_cpsc2021_protocol_gate.json',
    'experimental/external/ptbxl/ptbxl_full_predictions.npz',
    'experimental/external/ptbxl/ptbxl_full_slice_predictions.npz',
    'experimental/external/georgia/georgia_full_predictions.npz',
    'experimental/external/georgia/georgia_full_slice_predictions.npz',
    'experimental/external/cpsc2021/cpsc2021_full_predictions.npz',
    'experimental/external/cpsc2021/cpsc2021_full_slice_predictions.npz',
    'predictions/representation_embeddings_final_ema.npz',
    'manifests/representation_embedding_manifest.json',
    'metrics/representation_probe_summary.json',
    'metrics/representation_evidence_status.json',
    'tables/table_representation_probe.csv',
    'tables/table_representation_probe_by_fold.csv',
    'tables/table_representation_cka.csv',
    'tables/table_representation_evidence_status.csv',
    'figures/figure_representation_audit.png',
    'manifests/representation_probe_manifest.json',
    'metrics/physiological_interval_probe_summary.json',
    'tables/table_physiological_interval_probe.csv',
    'tables/table_physiological_interval_target_audit.csv',
    'tables/table_physiological_interval_probe_contrasts.csv',
    'manifests/physiological_interval_probe_manifest.json',
]

def restore_notebook06_artifacts(relative_paths):
    import shutil

    mirror_root = globals().get(
        'MIRROR_REVISION_ROOT',
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision',
    )
    manifest_path = mirror_root / 'manifests' / 'mirror_manifest.json'
    rows = {}
    if manifest_path.exists():
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
        rows = {row.get('relative_path'): row for row in payload.get('artifacts', []) if row.get('relative_path')}
        representation_cache_root = mirror_root / 'predictions' / 'folds'
        stale_representation_cache_rows = []
        for source in sorted(representation_cache_root.glob('representation_*.npz')):
            relative = source.relative_to(mirror_root).as_posix()
            row = rows.get(relative)
            if (
                row is None
                or int(row.get('size_bytes', -1) or -1) != source.stat().st_size
                or row.get('sha256') != sha256_file(source)
            ):
                stale_representation_cache_rows.append(relative)
        if stale_representation_cache_rows:
            print(
                'Ignoring unauthenticated/stale representation cache rows; '
                'the extraction runner must validate and republish them:',
                stale_representation_cache_rows,
            )
        for relative in rows:
            if (
                relative.startswith('predictions/folds/representation_')
                and relative.endswith('.npz')
                and relative not in stale_representation_cache_rows
            ):
                relative_paths.append(relative)

    restored, reused, unavailable = [], [], []
    for relative in sorted(set(relative_paths)):
        destination = revision_root / relative
        source = next(
            (root / relative for root in [mirror_root] if (root / relative).exists()),
            None,
        )
        if source is None or source.stat().st_size == 0:
            unavailable.append(relative)
            continue
        source_sha = sha256_file(source)
        row = rows.get(relative) if source.is_relative_to(mirror_root) else None
        if row is None:
            if destination.exists() and destination.stat().st_size > 0:
                raise RuntimeError(f'Active Notebook 06 artifact is absent from the canonical mirror manifest: {relative}')
            unavailable.append(relative)
            continue
        if row.get('sha256') != source_sha:
            raise RuntimeError(f'Mirror checksum mismatch for Notebook 06 artifact: {relative}')
        if destination.exists() and destination.stat().st_size > 0 and sha256_file(destination) == source_sha:
            reused.append(relative)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        if sha256_file(destination) != source_sha:
            raise RuntimeError(f'Checksum mismatch after Notebook 06 restore: {relative}')
        restored.append(relative)
    print(f'Notebook 06 targeted restore: restored={len(restored)} reused={len(reused)} unavailable={len(unavailable)}')
    if restored:
        print('Restored:', ', '.join(restored))

restore_notebook06_artifacts(RESTORE_RELATIVE_PATHS_FOR_06)

required_inputs = {
    'oof_final_ema_freeze_manifest': freeze_path,
    'frozen_final_ema_predictions': record_path,
    'frozen_final_ema_slice_predictions': slice_path,
    'calibration_ci_final_ema': calibration_path,
    'oof_final_ema_prediction_run_manifest': run_manifest_path,
}
for name, path in required_inputs.items():
    print(f'{name:34s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 02/03 final_ema artifacts are required before notebook 06: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before pooling sensitivity.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 06 requires checkpoint_kind=final_ema in the freeze manifest.')

frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
for artifact_path in [record_path, slice_path]:
    rel = artifact_path.as_posix()
    if rel not in frozen_artifacts:
        raise ValueError(f'Freeze manifest does not include {rel}')
    current_sha = sha256_file(artifact_path)
    if current_sha != frozen_artifacts[rel]['sha256']:
        raise RuntimeError(f'Frozen artifact checksum changed: {rel}')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'oof_stem': OOF_STEM,
    'threshold': THRESHOLD,
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'checkpoint_kind': freeze.get('checkpoint_kind'),
    'allowed_execution': {
        'pooling_sensitivity': RUN_POOLING_SENSITIVITY,
        'force_rerun_pooling_sensitivity': FORCE_RERUN_POOLING_SENSITIVITY,
        'representation_embedding_extraction': RUN_REPRESENTATION_EMBEDDING_EXTRACTION,
        'representation_probing': RUN_REPRESENTATION_PROBING,
        'model_training': False,
        'model_inference': bool(RUN_REPRESENTATION_EMBEDDING_EXTRACTION),
    },
    'representation_controls': {
        'only_folds': REPRESENTATION_ONLY_FOLDS,
        'batch_size': REPRESENTATION_BATCH_SIZE,
        'num_workers': REPRESENTATION_NUM_WORKERS,
        'probe_max_iter': REPRESENTATION_PROBE_MAX_ITER,
    },
}
save_json(revision_root / 'manifests' / 'pooling_representation_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'pooling_representation_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 06 to the tracked revision plan and claim map. It separates supported pooling analysis from still-blocked representation evidence.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_POOL', 'EXP_REPR'])]
relevant_claims = claims[claims['claim_id'].isin(['C04', 'C05'])]
relevant_tasks = tasks[tasks['id'].isin(['B1', 'B3'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'pooling_representation_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'pooling_representation_plan_alignment.json')


## Run Pooling Sensitivity

This uses only checksum-verified frozen OOF slice probabilities. It performs no model inference, no training, and no threshold tuning.


In [ ]:
pooling_command = (
    f'python -u scripts/revision/07_pooling_sensitivity.py '
    f'--freeze-manifest {freeze_path} '
    f'--record-file {record_path} '
    f'--slice-file {slice_path} '
    f'--expected-checkpoint-kind final_ema '
    f'--threshold {THRESHOLD}'
)
pooling_expected_outputs = [
    revision_root / 'metrics' / 'pooling_sensitivity.csv',
    revision_root / 'metrics' / 'pooling_sensitivity.json',
    revision_root / 'metrics' / 'pooling_decision_summary.json',
    revision_root / 'tables' / 'table_pooling_sensitivity.csv',
    revision_root / 'metrics' / 'pooling_sensitivity_across_datasets.csv',
    revision_root / 'tables' / 'table_pooling_sensitivity_across_datasets.csv',
    revision_root / 'metrics' / 'pooling_q3_paired_bootstrap.json',
    revision_root / 'manifests' / 'pooling_sensitivity_external_manifest.json',
]
def _pooling_artifacts_match_active_freeze():
    if not all(path.exists() and path.stat().st_size > 0 for path in pooling_expected_outputs):
        return False
    try:
        from scripts.revision.common import sha256_file
        detail = json.loads(pooling_expected_outputs[1].read_text(encoding='utf-8'))
        if detail.get('source_freeze_manifest_sha256') != sha256_file(freeze_path):
            return False
        if Path(detail.get('record_file', '')).name != record_path.name:
            return False
        if Path(detail.get('slice_file', '')).name != slice_path.name:
            return False
        if float(detail.get('threshold')) != float(THRESHOLD):
            return False
        expected_methods = {'mean', 'power_mean_q2', 'power_mean_q3', 'power_mean_q4', 'power_mean_q8', 'max'}
        summary_df = pd.read_csv(pooling_expected_outputs[0])
        table_df = pd.read_csv(pooling_expected_outputs[3])
        if set(summary_df.get('pooling', pd.Series(dtype=str)).astype(str)) != expected_methods:
            return False
        if summary_df.shape != table_df.shape or list(summary_df.columns) != list(table_df.columns):
            return False
        if not summary_df.equals(table_df):
            return False
        return True
    except Exception as exc:
        print('Pooling sensitivity reuse contract rejected:', repr(exc))
        return False

pooling_artifacts_ready = _pooling_artifacts_match_active_freeze()
pooling_sensitivity_ran = False
if RUN_POOLING_SENSITIVITY and (FORCE_RERUN_POOLING_SENSITIVITY or not pooling_artifacts_ready):
    run(pooling_command, log_path='reports/revision/logs/oof_final_ema_pooling_sensitivity.log')
    pooling_sensitivity_ran = True
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --refresh-existing-prefix predictions/folds --mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/pooling_sensitivity_immediate_mirror_publish.log',
    )
elif pooling_artifacts_ready:
    print('Reusing existing pooling sensitivity artifacts. Set FORCE_RERUN_POOLING_SENSITIVITY=True to recompute.')
else:
    print('Pooling sensitivity disabled. Planned command:', pooling_command)


## Inspect Pooling Results And Decide Q=3 Wording

The decision artifact records whether Q=3 is best, competitive, or should be described only as the frozen/default aggregation setting.


In [ ]:
pooling_csv = revision_root / 'metrics' / 'pooling_sensitivity.csv'
pooling_json = revision_root / 'metrics' / 'pooling_sensitivity.json'
if not pooling_csv.exists() or not pooling_json.exists():
    raise FileNotFoundError('Pooling sensitivity outputs are missing; run the previous cell first.')

pooling_df = pd.read_csv(pooling_csv)
expected_methods = {'mean', 'power_mean_q2', 'power_mean_q3', 'power_mean_q4', 'power_mean_q8', 'max'}
observed_methods = set(pooling_df['pooling'].astype(str))
missing_methods = sorted(expected_methods - observed_methods)
if missing_methods:
    raise ValueError('Pooling sensitivity is incomplete; missing methods: ' + ', '.join(missing_methods))

metric_priority = ['roc_auc_macro', 'pr_auc_macro', 'f1_macro', 'recall_macro', 'specificity_macro']
q3 = pooling_df[pooling_df['pooling'] == 'power_mean_q3'].iloc[0].to_dict()
rank_summary = {}
for metric in metric_priority:
    ordered = pooling_df.sort_values(metric, ascending=False).reset_index(drop=True)
    rank = int(ordered.index[ordered['pooling'] == 'power_mean_q3'][0] + 1)
    best = ordered.iloc[0].to_dict()
    rank_summary[metric] = {
        'q3_rank': rank,
        'q3_value': float(q3[metric]),
        'best_pooling': best['pooling'],
        'best_value': float(best[metric]),
    }

q3_best_metrics = [metric for metric, row in rank_summary.items() if row['q3_rank'] == 1]
if q3_best_metrics:
    q3_decision = 'q3_supported_for_metrics_' + '_'.join(q3_best_metrics)
    safe_wording = 'Q=3 is supported for the listed metrics but should still be reported with the full sensitivity table.'
else:
    q3_decision = 'q3_not_best_present_as_frozen_default_or_tradeoff'
    safe_wording = 'Do not claim Q=3 is optimal; present it as the frozen/default aggregation and report the better pooling alternatives.'

decision = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_pooling_csv': str(pooling_csv),
    'source_pooling_json': str(pooling_json),
    'source_pooling_csv_sha256': sha256_file(pooling_csv),
    'threshold': THRESHOLD,
    'expected_methods': sorted(expected_methods),
    'rank_summary': rank_summary,
    'q3_decision': q3_decision,
    'safe_wording': safe_wording,
}
save_json(revision_root / 'metrics' / 'pooling_decision_summary.json', decision)
pooling_table = revision_root / 'tables' / 'table_pooling_sensitivity.csv'
pooling_df.to_csv(pooling_table, index=False)
print('Wrote:', revision_root / 'metrics' / 'pooling_decision_summary.json')
print('Wrote:', pooling_table)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --refresh-existing-prefix predictions/folds --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/pooling_decision_immediate_mirror_publish.log',
)
display(pooling_df)
print('Q=3 decision:', q3_decision)
print('Safe wording:', safe_wording)


## External Pooling Sensitivity


In [ ]:
# CPU-only. PTB-XL/Georgia are mapped record tasks; CPSC2021 remains a separate mapped-window task and is never pooled with them.
import hashlib
RUN_EXTERNAL_POOLING_SENSITIVITY = 'auto'
EXTERNAL_POOLING_DATASETS = 'ptbxl,georgia,cpsc2021'
EXTERNAL_POOLING_N_BOOT = 1000
EXTERNAL_POOLING_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'pooling_external_metric_cache'
external_pooling_required = [
    revision_root / 'metrics' / 'pooling_sensitivity_external.csv',
    revision_root / 'tables' / 'table_pooling_sensitivity_across_datasets.csv',
    revision_root / 'metrics' / 'pooling_q3_paired_bootstrap.json',
    revision_root / 'manifests' / 'pooling_sensitivity_external_manifest.json',
]
external_pooling_datasets = [item.strip() for item in EXTERNAL_POOLING_DATASETS.split(',') if item.strip()]

def _pooling_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _external_pooling_contract_ready():
    if not all(path.exists() and path.stat().st_size > 0 for path in external_pooling_required):
        return False
    manifest_path = external_pooling_required[-1]
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        expected_runner = Path('scripts/revision/30_pooling_sensitivity_external.py')
        if manifest.get('status') is not True:
            return False
        if manifest.get('protocol') != 'external_pooling_sensitivity_v2_group_bootstrap':
            return False
        if manifest.get('datasets') != external_pooling_datasets:
            return False
        if manifest.get('methods') != ['mean', 'power_mean_q2', 'power_mean_q3', 'power_mean_q4', 'power_mean_q8', 'max']:
            return False
        if manifest.get('threshold') != 0.5 or manifest.get('n_bins') != 15:
            return False
        if manifest.get('n_boot') != EXTERNAL_POOLING_N_BOOT or manifest.get('seed') != 42:
            return False
        if manifest.get('strict_group_bootstrap') is not True:
            return False
        if manifest.get('runner_sha256') != _pooling_sha256(expected_runner):
            return False
        input_rows = {(row.get('dataset'), row.get('kind')): row for row in manifest.get('inputs', [])}
        for dataset in external_pooling_datasets:
            root = revision_root / 'experimental' / 'external' / dataset
            expected_inputs = {
                'protocol_gate': revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
                'record_predictions': root / f'{dataset}_full_predictions.npz',
                'slice_predictions': root / f'{dataset}_full_slice_predictions.npz',
            }
            for kind, path in expected_inputs.items():
                row = input_rows.get((dataset, kind))
                if row is None or not path.exists() or row.get('sha256') != _pooling_sha256(path):
                    return False
        output_rows = {Path(row.get('path', '')).name: row for row in manifest.get('outputs', [])}
        for path in external_pooling_required[:-1]:
            row = output_rows.get(path.name)
            if row is None or row.get('sha256') != _pooling_sha256(path):
                return False
        return True
    except Exception as exc:
        print(f'External pooling reuse contract rejected: {exc}')
        return False

external_pooling_ready = _external_pooling_contract_ready()
external_pooling_should_run = RUN_EXTERNAL_POOLING_SENSITIVITY is True or (
    str(RUN_EXTERNAL_POOLING_SENSITIVITY).lower() == 'auto' and not external_pooling_ready
)
external_pooling_sources = [
    path
    for dataset in external_pooling_datasets
    for path in (
        revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
        revision_root / 'experimental' / 'external' / dataset / f'{dataset}_full_predictions.npz',
        revision_root / 'experimental' / 'external' / dataset / f'{dataset}_full_slice_predictions.npz',
    )
]
missing_external_pooling_sources = [
    str(path) for path in external_pooling_sources if not path.exists() or path.stat().st_size == 0
]
if missing_external_pooling_sources:
    if RUN_EXTERNAL_POOLING_SENSITIVITY is True:
        raise FileNotFoundError(
            'External pooling was forced, but source artifacts are missing: ' + '; '.join(missing_external_pooling_sources)
        )
    external_pooling_should_run = False
    print('Deferred external pooling until Notebook 02 protocol-gated PTB-XL/Georgia/CPSC2021 predictions are complete.')
external_pooling_ran = False
if external_pooling_should_run:
    run(
        'python -u scripts/revision/30_pooling_sensitivity_external.py '
        + ' '.join(f'--dataset {item}' for item in external_pooling_datasets)
        + f' --threshold 0.5 --n-bins 15 --n-boot {EXTERNAL_POOLING_N_BOOT} '
        + f'--strict-group-bootstrap --reuse-existing --metric-cache-dir "{EXTERNAL_POOLING_CACHE_DIR}"',
        log_path='reports/revision/logs/external_pooling_sensitivity.log',
    )
    external_pooling_ran = True
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --source-conflict-policy source --refresh-existing-cache-dirs --refresh-existing-prefix predictions/folds --refresh-existing-prefix metrics/pooling_external_metric_cache --mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/external_pooling_immediate_mirror_publish.log',
    )
else:
    print('Reusing external pooling sensitivity artifacts.' if external_pooling_ready else 'External pooling deferred; source or output artifacts are missing/stale.')
for path in external_pooling_required:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Representation Embeddings And Probe/CKA

Optional representation evidence now has an implemented extraction hook and probe runner. Keep both flags `False` unless you intentionally want to run model inference/probing. The extractor writes fold-level caches so interrupted Colab sessions can resume with `REPRESENTATION_ONLY_FOLDS`.

Safe claim boundary: results may support branch-specific information patterns only; they do not establish strict morphology-rhythm separation.


### Representation Runtime Guard

Run this guard before representation extraction. It verifies or installs `mamba_ssm` and `causal_conv1d` in the current GPU runtime. If you pulled a newer repo while this Colab tab was already open, reopen the notebook file so the browser uses the refreshed cells.


In [ ]:
REPRESENTATION_NOTEBOOK_RUNTIME_REVISION = '2026-07-16-mamba-installer-contract-v4'
print('Notebook 06 representation runtime guard revision:', REPRESENTATION_NOTEBOOK_RUNTIME_REVISION)

def ensure_mamba_runtime_for_representation():
    import importlib
    import importlib.util
    import json as _json

    required_modules = ['mamba_ssm', 'causal_conv1d']
    missing = [module for module in required_modules if importlib.util.find_spec(module) is None]
    if not missing:
        print('Mamba runtime available for representation extraction.')
        return
    installer_nb_path = REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb'
    if not installer_nb_path.exists():
        raise FileNotFoundError('Missing canonical Mamba installer notebook: ' + str(installer_nb_path))
    installer_nb = _json.loads(installer_nb_path.read_text(encoding='utf-8'))
    installer_candidates = []
    for cell_index, cell in enumerate(installer_nb['cells']):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        if "MAMBA_INSTALLER_CAPABILITY = 'ecg_ramba_mamba_installer_v1'" in source and 'MAMBA_INSTALLER_SCHEMA_VERSION = 1' in source:
            installer_candidates.append((cell_index, source))
    installer_source = installer_candidates[0][1] if len(installer_candidates) == 1 else None
    if installer_source is None:
        raise RuntimeError(
            'Could not uniquely locate the canonical Mamba installer cell in Notebook 02; '
            f'candidate_count={len(installer_candidates)}.'
        )
    print('Canonical Mamba installer source: Notebook 02 cell', installer_candidates[0][0])
    print('Installing the canonical Mamba runtime required for representation extraction...')
    exec(compile(installer_source, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
    importlib.invalidate_caches()
    still_missing = [module for module in required_modules if importlib.util.find_spec(module) is None]
    if still_missing:
        raise ImportError('Mamba runtime remains unavailable: ' + ', '.join(still_missing) + '. Restart only if the installer requests it, then rerun Notebook 06 from Setup.')
    print('Mamba runtime installed and importable for representation extraction.')


In [ ]:
embedding_path = revision_root / 'predictions' / 'representation_embeddings_final_ema.npz'
run_manifest_path = revision_root / 'manifests' / 'oof_final_ema_prediction_run_manifest.json'
embedding_manifest = revision_root / 'manifests' / 'representation_embedding_manifest.json'
probe_summary = revision_root / 'metrics' / 'representation_probe_summary.json'
probe_table = revision_root / 'tables' / 'table_representation_probe.csv'
fold_probe_table = revision_root / 'tables' / 'table_representation_probe_by_fold.csv'
cka_table = revision_root / 'tables' / 'table_representation_cka.csv'
representation_figure = revision_root / 'figures' / 'figure_representation_audit.png'
probe_manifest = revision_root / 'manifests' / 'representation_probe_manifest.json'
v3_protocol = 'representation_probe_fold_safe_v3_projection_and_fold_audit'

def _ready(paths):
    return all(path.exists() and path.stat().st_size > 0 for path in paths)

def _active_representation_contract():
    from scripts.revision.common import sha256_file
    return {
        'oof_sha256': sha256_file(record_path),
        'freeze_sha256': sha256_file(freeze_path),
    }

def _embedding_is_current():
    if not _ready([embedding_path, embedding_manifest]):
        return False
    try:
        from scripts.revision.common import sha256_file
        payload = json.loads(embedding_manifest.read_text(encoding='utf-8'))
        return bool(
            payload.get('status') == 'complete'
            and payload.get('protocol') == 'ecg_ramba_final_ema_branch_embedding_extraction_v1'
            and payload.get('canonical_contract') == _active_representation_contract()
            and payload.get('runner_sha256') == sha256_file(Path('scripts/revision/22_extract_representations.py'))
            and (payload.get('outputs') or {}).get('embedding_npz_sha256') == sha256_file(embedding_path)
        )
    except Exception as exc:
        print('Representation embedding reuse contract rejected:', repr(exc))
        return False

embedding_refresh_audit = None

def _embedding_npz_can_refresh_manifest_without_inference():
    global embedding_refresh_audit
    if not embedding_path.exists() or embedding_path.stat().st_size == 0:
        return False
    try:
        import importlib
        extraction_runner = importlib.import_module('scripts.revision.22_extract_representations')
        active_oof = extraction_runner.load_oof_contract(record_path, freeze_path, 0)
        checkpoint_contracts = extraction_runner.load_checkpoint_contracts(run_manifest_path, 'final_ema')
        checkpoint_split_contract = extraction_runner.validate_checkpoint_fold_contract(
            active_oof, checkpoint_contracts
        )
        reuse_audit = extraction_runner.inspect_final_embedding_reuse(
            embedding_path, active_oof, 'final_ema', checkpoint_contracts
        )
        reuse_audit['checkpoint_split_contract'] = checkpoint_split_contract
        embedding_refresh_audit = reuse_audit
        print('Representation embedding semantic refresh audit:', reuse_audit)
        return bool(reuse_audit.get('reusable'))
    except Exception as exc:
        embedding_refresh_audit = {'reusable': False, 'issues': [f'{type(exc).__name__}: {exc}']}
        print('Representation embedding NPZ cannot be reused:', repr(exc))
        return False

def _probe_is_v3_ready():
    required = [probe_summary, probe_table, fold_probe_table, cka_table, representation_figure, probe_manifest]
    if not _ready(required):
        return False
    try:
        from scripts.revision.common import sha256_file
        payload = json.loads(probe_manifest.read_text(encoding='utf-8'))
        artifact_sha = payload.get('artifact_sha256') or {}
        return bool(
            payload.get('status') == 'complete'
            and payload.get('protocol') == v3_protocol
            and payload.get('canonical_contract') == _active_representation_contract()
            and payload.get('runner_sha256') == sha256_file(Path('scripts/revision/20_representation_probe.py'))
            and (payload.get('embedding_npz') or {}).get('sha256') == sha256_file(embedding_path)
            and artifact_sha.get('probe_table') == sha256_file(probe_table)
            and artifact_sha.get('fold_probe_table') == sha256_file(fold_probe_table)
            and artifact_sha.get('cka_table') == sha256_file(cka_table)
            and artifact_sha.get('audit_figure') == sha256_file(representation_figure)
        )
    except Exception as exc:
        print('Representation probe reuse contract rejected:', repr(exc))
        return False

embedding_ready = _embedding_is_current()
probe_ready = _probe_is_v3_ready()
extraction_should_run = RUN_REPRESENTATION_EMBEDDING_EXTRACTION is True or (
    str(RUN_REPRESENTATION_EMBEDDING_EXTRACTION).lower() == 'auto' and not embedding_ready
)
probing_should_run = RUN_REPRESENTATION_PROBING is True or (
    str(RUN_REPRESENTATION_PROBING).lower() == 'auto' and not probe_ready
)
extract_command = (
    'python -u scripts/revision/22_extract_representations.py '
    '--checkpoint-kind final_ema '
    f'--oof-predictions {record_path} '
    f'--freeze-manifest {freeze_path} '
    f'--oof-run-manifest {run_manifest_path} '
    f'--out-embedding {embedding_path} '
    f'--out-manifest {embedding_manifest} '
    f'--batch-size {REPRESENTATION_BATCH_SIZE} '
    f'--num-workers {REPRESENTATION_NUM_WORKERS} '
    f'--fold-cache-dir "{REPRESENTATION_FOLD_CACHE_DIR}"'
)
if REPRESENTATION_ONLY_FOLDS.strip():
    extract_command += f' --only-folds "{REPRESENTATION_ONLY_FOLDS}"'
probe_command = (
    'python -u scripts/revision/20_representation_probe.py '
    f'--embedding-npz {embedding_path} '
    f'--out-summary {probe_summary} '
    f'--out-probe-table {probe_table} '
    f'--out-fold-probe-table {fold_probe_table} '
    f'--out-cka-table {cka_table} '
    f'--out-audit-figure {representation_figure} '
    f'--out-manifest {probe_manifest} '
    f'--probe-max-iter {REPRESENTATION_PROBE_MAX_ITER}'
)
print('Representation extraction command:', extract_command)
print('Representation probe command     :', probe_command)
print('Embedding reusable:', embedding_ready, '| Probe v3 reusable:', probe_ready)

if extraction_should_run:
    manifest_refresh_only = _embedding_npz_can_refresh_manifest_without_inference()
    if not manifest_refresh_only:
        import torch
        if not torch.cuda.is_available():
            issues = (embedding_refresh_audit or {}).get('issues', ['embedding_not_reusable'])
            field_match = (embedding_refresh_audit or {}).get('semantic_field_match', {})
            raise RuntimeError(
                'Representation extraction requires an A100/GPU runtime because the existing embedding cannot be safely reused. '
                f'issues={issues}; semantic_field_match={field_match}. '
                'The extractor now uses the fold_id frozen in the canonical OOF artifact and will reject old per-fold caches '
                'whose validation membership differs. Install the canonical Mamba runtime, then rerun this cell.'
            )
        ensure_mamba_runtime_for_representation()
    else:
        print('Refreshing representation provenance from the verified final embedding; GPU/Mamba is not needed.')
    run(extract_command, log_path='reports/revision/logs/representation_embedding_extraction.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --refresh-existing-prefix predictions/folds --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/representation_embedding_immediate_mirror_publish.log',
        )
    embedding_ready = _embedding_is_current()
else:
    print('Reusing representation embedding artifact and fold caches.')

if probing_should_run:
    if not embedding_ready:
        raise FileNotFoundError('Representation embedding NPZ is missing; complete extraction before the probe.')
    run(probe_command, log_path='reports/revision/logs/representation_probe.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --refresh-existing-prefix predictions/folds --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/representation_probe_immediate_mirror_publish.log',
        )
    probe_ready = _probe_is_v3_ready()
else:
    print('Reusing v3 fold-safe representation probe/CKA/figure artifacts.')

representation_outputs = {
    'embedding_npz': embedding_path,
    'embedding_manifest': embedding_manifest,
    'probe_summary': probe_summary,
    'probe_table': probe_table,
    'fold_probe_table': fold_probe_table,
    'cka_table': cka_table,
    'audit_figure': representation_figure,
    'probe_manifest': probe_manifest,
}
representation_rows = [
    {
        'evidence_name': name,
        'path': str(path),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
        'status': 'complete' if path.exists() and path.stat().st_size > 0 else 'missing',
        'sha256': sha256_file(path) if path.exists() and path.stat().st_size > 0 else '',
    }
    for name, path in representation_outputs.items()
]
if probe_ready:
    representation_status = 'complete_probe_available_with_disentanglement_limitation'
    safe_wording = 'Fold-safe probe and CKA results audit representation similarity only; they do not support label-aligned morphology-rhythm disentanglement.'
elif embedding_ready:
    representation_status = 'embeddings_ready_probe_not_run_or_stale'
    safe_wording = 'Embedding artifacts exist, but the required v3 fold-safe probe audit is incomplete or stale; do not claim representation separation.'
else:
    representation_status = 'blocked_missing_embeddings'
    safe_wording = 'Do not claim representation separation; run the frozen-model embedding extractor and v3 fold-safe probe first.'
repr_df = pd.DataFrame(representation_rows)
repr_table = revision_root / 'tables' / 'table_representation_evidence_status.csv'
repr_json = revision_root / 'metrics' / 'representation_evidence_status.json'
repr_df.to_csv(repr_table, index=False)
save_json(repr_json, {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'run_representation_embedding_extraction': RUN_REPRESENTATION_EMBEDDING_EXTRACTION,
    'run_representation_probing': RUN_REPRESENTATION_PROBING,
    'status': representation_status,
    'safe_wording': safe_wording,
    'extract_command': extract_command,
    'probe_command': probe_command,
    'rows': representation_rows,
})
print('Wrote:', repr_table)
print('Wrote:', repr_json)
print('Representation status:', representation_status)
display(repr_df)


## Measured Physiological Interval Probe Gate


In [ ]:
# CPU-only once representation embeddings exist. No proxy interval targets are manufactured.
PHYSIOLOGICAL_PROBE_N_BOOT = 1000
RUN_PHYSIOLOGICAL_INTERVAL_PROBE = 'auto'

physiology_relative_outputs = [
    'metrics/physiological_interval_probe_summary.json',
    'tables/table_physiological_interval_probe.csv',
    'tables/table_physiological_interval_target_audit.csv',
    'tables/table_physiological_interval_probe_contrasts.csv',
    'tables/table_physiological_interval_probe.tex',
    'manifests/physiological_interval_probe_manifest.json',
]
restore_notebook06_artifacts(physiology_relative_outputs)

metadata_candidates = [
    Path(os.environ['ECG_RAMBA_PHYSIOLOGY_METADATA']) if os.environ.get('ECG_RAMBA_PHYSIOLOGY_METADATA') else None,
    DRIVE_ROOT / 'physiological_interval_metadata.csv',
    DRIVE_ROOT / 'metadata' / 'physiological_interval_metadata.csv',
]
physiology_metadata = next((path for path in metadata_candidates if path is not None and path.is_file()), None)
physiology_provenance = None
if physiology_metadata is not None:
    print('Physiological metadata candidate:', physiology_metadata)
    print('Physiological metadata SHA256 for provenance sidecar:', sha256_file(physiology_metadata))
    print(
        'The reviewed sidecar must bind this SHA and set independent_of_model_outputs=true and '
        'independent_of_ecg_ramba_feature_cache=true.'
    )
    provenance_candidates = [
        Path(os.environ['ECG_RAMBA_PHYSIOLOGY_PROVENANCE'])
        if os.environ.get('ECG_RAMBA_PHYSIOLOGY_PROVENANCE') else None,
        physiology_metadata.with_suffix(physiology_metadata.suffix + '.provenance.json'),
        physiology_metadata.with_name(physiology_metadata.stem + '_provenance.json'),
    ]
    physiology_provenance = next(
        (path for path in provenance_candidates if path is not None and path.is_file()), None
    )
physiology_command = (
    'python -u scripts/revision/44_physiological_interval_probe.py '
    '--embedding-npz reports/revision/predictions/representation_embeddings_final_ema.npz '
    '--embedding-manifest reports/revision/manifests/representation_embedding_manifest.json '
    f'--n-boot {PHYSIOLOGICAL_PROBE_N_BOOT}'
)
if physiology_metadata is not None:
    physiology_command += f' --metadata-csv "{physiology_metadata}"'
    if physiology_provenance is not None:
        physiology_command += f' --metadata-provenance-json "{physiology_provenance}"'
    else:
        print('Measured metadata found but no reviewed provenance sidecar found; the runner will emit a blocker artifact.')
else:
    print('No reliable measured HR/PR/QRS/QT/QTc metadata CSV found; the runner will emit a blocker artifact.')

physiology_summary_path = Path('reports/revision/metrics/physiological_interval_probe_summary.json')
physiology_manifest_path = Path('reports/revision/manifests/physiological_interval_probe_manifest.json')
physiology_runner_path = Path('scripts/revision/44_physiological_interval_probe.py')
physiology_embedding_path = Path('reports/revision/predictions/representation_embeddings_final_ema.npz')
physiology_embedding_manifest_path = Path('reports/revision/manifests/representation_embedding_manifest.json')
physiology_reusable = False
if physiology_summary_path.is_file() and physiology_manifest_path.is_file():
    existing_summary = json.loads(physiology_summary_path.read_text(encoding='utf-8'))
    existing_manifest = json.loads(physiology_manifest_path.read_text(encoding='utf-8'))
    existing_inputs = existing_manifest.get('inputs') or {}
    existing_outputs = existing_manifest.get('outputs') or {}
    physiology_reusable = (
        existing_summary.get('protocol') == 'fold_held_out_measured_physiological_interval_probe_v3'
        and existing_manifest.get('protocol') == 'fold_held_out_measured_physiological_interval_probe_v3'
        and (existing_manifest.get('runner') or {}).get('sha256') == sha256_file(physiology_runner_path)
        and (existing_inputs.get('embedding') or {}).get('sha256') == sha256_file(physiology_embedding_path)
        and (existing_inputs.get('embedding_manifest') or {}).get('sha256')
        == sha256_file(physiology_embedding_manifest_path)
        and existing_manifest.get('status') == existing_summary.get('status')
        and existing_summary.get('status') in {
            'complete_measured_target_probe',
            'blocked_missing_reliable_interval_metadata',
        }
    )
    physiology_common_output_relatives = [
        relative for relative in physiology_relative_outputs
        if not relative.endswith('.tex') and not relative.startswith('manifests/')
    ]
    physiology_expected_output_relatives = (
        physiology_common_output_relatives + ['tables/table_physiological_interval_probe.tex']
        if existing_summary.get('status') == 'complete_measured_target_probe'
        else physiology_common_output_relatives
    )
    authenticated_output_paths = [
        Path('reports/revision') / relative for relative in physiology_expected_output_relatives
    ]
    physiology_reusable = physiology_reusable and all(
        path.exists()
        and existing_outputs.get(path.as_posix()) == sha256_file(path)
        for path in authenticated_output_paths
    )
    if physiology_metadata is None:
        physiology_reusable = physiology_reusable and existing_summary.get('status') == 'blocked_missing_reliable_interval_metadata'
    elif physiology_provenance is None:
        physiology_reusable = (
            physiology_reusable
            and (existing_inputs.get('metadata') or {}).get('sha256') == sha256_file(physiology_metadata)
            and existing_summary.get('status') == 'blocked_missing_reliable_interval_metadata'
        )
    else:
        physiology_reusable = (
            physiology_reusable
            and (existing_inputs.get('metadata') or {}).get('sha256') == sha256_file(physiology_metadata)
            and (existing_inputs.get('metadata_provenance') or {}).get('sha256') == sha256_file(physiology_provenance)
        )
run_physiology_probe = (
    not physiology_reusable
    if str(RUN_PHYSIOLOGICAL_INTERVAL_PROBE).lower() == 'auto'
    else bool(RUN_PHYSIOLOGICAL_INTERVAL_PROBE)
)
print('Physiological probe reusable:', physiology_reusable, '| should_run=', run_physiology_probe)
print('Physiological probe command:', physiology_command)
if run_physiology_probe:
    run(
        physiology_command,
        log_path='reports/revision/logs/physiological_interval_probe.log',
    )
    run(
        f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
        f'--refresh-existing-prefix metrics/physiological_interval_probe_summary.json '
        f'--refresh-existing-prefix tables/table_physiological_interval_ '
        f'--refresh-existing-prefix manifests/physiological_interval_probe_manifest.json '
        f'--mirror-root "{MIRROR_REVISION_ROOT}"',
        log_path='reports/revision/logs/physiological_interval_probe_mirror_publish.log',
    )

if not physiology_summary_path.is_file():
    raise FileNotFoundError(physiology_summary_path)
physiology_summary = json.loads(physiology_summary_path.read_text(encoding='utf-8'))
print(json.dumps({
    'status': physiology_summary.get('status'),
    'targets': physiology_summary.get('targets', []),
    'claim_boundary': physiology_summary.get('claim_boundary'),
}, indent=2))


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/pooling_representation_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --refresh-existing-prefix predictions/folds --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/pooling_representation_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'pooling_representation_input_contract.json',
    revision_root / 'manifests' / 'pooling_representation_plan_alignment.json',
    revision_root / 'metrics' / 'pooling_sensitivity.csv',
    revision_root / 'metrics' / 'pooling_sensitivity.json',
    revision_root / 'metrics' / 'pooling_decision_summary.json',
    revision_root / 'tables' / 'table_pooling_sensitivity.csv',
    revision_root / 'metrics' / 'representation_evidence_status.json',
    revision_root / 'tables' / 'table_representation_evidence_status.csv',
    revision_root / 'predictions' / 'representation_embeddings_final_ema.npz',
    revision_root / 'manifests' / 'representation_embedding_manifest.json',
    revision_root / 'metrics' / 'representation_probe_summary.json',
    revision_root / 'tables' / 'table_representation_probe.csv',
    revision_root / 'tables' / 'table_representation_probe_by_fold.csv',
    revision_root / 'tables' / 'table_representation_cka.csv',
    revision_root / 'figures' / 'figure_representation_audit.png',
    revision_root / 'manifests' / 'representation_probe_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
if not _pooling_artifacts_match_active_freeze():
    raise RuntimeError('Notebook 06 pooling package is missing or stale for the active final_ema freeze contract.')
external_pooling_manifest_path = revision_root / 'manifests' / 'pooling_sensitivity_external_manifest.json'
if not external_pooling_manifest_path.exists() or external_pooling_manifest_path.stat().st_size == 0:
    raise FileNotFoundError('Cross-dataset pooling manifest is missing; run External Pooling Sensitivity.')
external_pooling_manifest = json.loads(external_pooling_manifest_path.read_text(encoding='utf-8'))
if (
    external_pooling_manifest.get('status') is not True
    or external_pooling_manifest.get('protocol') != 'external_pooling_sensitivity_v2_group_bootstrap'
    or external_pooling_manifest.get('datasets') != ['ptbxl', 'georgia', 'cpsc2021']
    or int(external_pooling_manifest.get('n_boot', 0)) != 1000
    or external_pooling_manifest.get('strict_group_bootstrap') is not True
):
    raise RuntimeError('Cross-dataset pooling package does not satisfy the 3-dataset strict group-bootstrap contract.')
if str(RUN_REPRESENTATION_EMBEDDING_EXTRACTION).lower() in {'true', 'auto'} and not _embedding_is_current():
    raise RuntimeError('Notebook 06 representation embedding package is missing or stale after execution.')
if str(RUN_REPRESENTATION_PROBING).lower() in {'true', 'auto'} and not _probe_is_v3_ready():
    raise RuntimeError('Notebook 06 v3 representation probe/CKA/figure package is missing or stale after execution.')
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
print('Notebook 06 complete. Pooling sensitivity can support/revise Q=3 wording. Representation claims require completed embedding/probe artifacts and remain limited to suggestive branch-specific evidence.')
